# ** Preprocessing **

In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# I/ Users preprocessing

In [206]:
users = pd.read_csv("../data/users_dataset.csv")

In [207]:
users.columns

Index(['user_id', 'age', 'sexe', 'situation_familiale', 'nombre_enfants',
       'revenu_mensuel', 'budget_max', 'usage_principal',
       'distance_journaliere_km', 'prefere_electrique', 'carrosserie_preferee',
       'energie_preferee', 'boite_preferee', 'importance_consommation',
       'importance_performance', 'importance_confort', 'importance_prix'],
      dtype='object')

In [208]:
users.head()

,user_id,age,sexe,situation_familiale,nombre_enfants,revenu_mensuel,budget_max,usage_principal,distance_journaliere_km,prefere_electrique,carrosserie_preferee,energie_preferee,boite_preferee,importance_consommation,importance_performance,importance_confort,importance_prix
0,1,62,Homme,Célibataire,0,1319,40000,Plage,134,0,SUV,Hybride,Automatique,1,1,2,2
1,2,57,Homme,Marié(e),2,2739,73364,Sport,91,0,Coupé,Diesel,Manuelle,3,3,2,2
2,3,27,Femme,Célibataire,0,1077,40000,Plage,54,0,SUV,Hybride,Automatique,5,3,5,3
3,4,24,Homme,Marié(e),4,5155,116516,Désert,23,1,Minibus,Electrique,Automatique,3,3,2,3
4,5,60,Homme,Marié(e),1,2005,69754,Montagne,73,1,SUV,Electrique,Automatique,2,1,3,4


In [209]:
users.shape

(1000, 17)

###  # Data cleaning

In [210]:
users = users.drop("user_id", axis=1)

In [211]:
# Strip(remove whitespace) & Lowercase Text Columns
text_cols = [
    "sexe",
    "situation_familiale",
    "usage_principal",
    "carrosserie_preferee",
    "energie_preferee",
    "boite_preferee"
]


for col in text_cols:
    users[col] = users[col].str.strip().str.lower()


In [212]:
# Remove Duplicates
users = users.drop_duplicates()

###  # Feature Engineering

In [213]:
# Family Features
users["min_required_seats"] = users["nombre_enfants"] + 1
users["has_children"] = (users["nombre_enfants"] > 0).astype(int)

# Long Commute
users["long_commute"] = (users["distance_journaliere_km"] > 50).astype(int)

# Needs AWD
users["needs_awd"] = users["usage_principal"].isin(
    ["désert", "montagne", "campagne"]
).astype(int)



###  # Encoding

In [214]:
# Binary Encoding

users["sexe"] = users["sexe"].map({
    "homme": 0,
    "femme": 1
})

users["boite_preferee"] = users["boite_preferee"].map({
    "manuelle": 0,
    "automatique": 1
})

# One-Hot Encoding

users = pd.get_dummies(users, columns=["situation_familiale"], prefix="situation_familiale")

users = pd.get_dummies(users, columns=["carrosserie_preferee"], prefix="carrosserie")

users = pd.get_dummies(users, columns=["energie_preferee"], prefix="energie")

users = pd.get_dummies(users, columns=["usage_principal"], prefix="usage")


# Convert all bool columns to int

bool_cols = users.select_dtypes(include="bool").columns
users[bool_cols] = users[bool_cols].astype(int)

###  # Scaling

In [215]:
# Columns that should be scaled (continuous values)
user_scale_cols = [
    "age",
    "revenu_mensuel",
    "budget_max",
    "distance_journaliere_km",
    "min_required_seats",
    "nombre_enfants",
]

# Importance scores (these are ordinal 1-5, should be scaled)
importance_cols = [
    "importance_consommation",  
    "importance_performance",
    "importance_confort", 
    "importance_prix"
]

# Binary columns (should NOT be scaled)
binary_cols = [
    'sexe',
    'prefere_electrique',
    'boite_preferee',
    'has_children',
    'long_commute',
    'needs_awd',
    'situation_familiale_célibataire',
    'situation_familiale_divorcé(e)',
    'situation_familiale_marié(e)',
    'carrosserie_berline',
    'carrosserie_cabriolet',
    'carrosserie_citadine',
    'carrosserie_compacte',
    'carrosserie_coupé',
    'carrosserie_minibus',
    'carrosserie_monospace',
    'carrosserie_pick up',
    'carrosserie_suv',
    'carrosserie_utilitaire',
    'energie_diesel',
    'energie_electrique',
    'energie_essence',
    'energie_hybride',
    'usage_autoroute',
    'usage_campagne',
    'usage_désert',
    'usage_montagne',
    'usage_plage',
    'usage_sport',
    'usage_ville'
]

# Combine all columns to scale
all_scale_cols = user_scale_cols + importance_cols

# Scale only these columns
scaler = MinMaxScaler()
users[all_scale_cols] = scaler.fit_transform(users[all_scale_cols])

print(f"Scaled {len(all_scale_cols)} numerical columns")
print(f"Preserved {len(binary_cols)} binary columns as 0/1")

# Verify
print("\nSample of scaled columns:")
for col in all_scale_cols[:3]:
    print(f"   {col}: {users[col].min():.3f} to {users[col].max():.3f}")

print("\nSample of binary columns:")
for col in binary_cols[:3]:
    print(f"   {col}: values {users[col].unique()}")

Scaled 10 numerical columns
Preserved 30 binary columns as 0/1

Sample of scaled columns:
   age: 0.000 to 1.000
   revenu_mensuel: 0.000 to 1.000
   budget_max: 0.000 to 1.000

Sample of binary columns:
   sexe: values [0 1]
   prefere_electrique: values [0 1]
   boite_preferee: values [1 0]


###  # Final Check

In [216]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   age                              1000 non-null   float64
 1   sexe                             1000 non-null   int64  
 2   nombre_enfants                   1000 non-null   float64
 3   revenu_mensuel                   1000 non-null   float64
 4   budget_max                       1000 non-null   float64
 5   distance_journaliere_km          1000 non-null   float64
 6   prefere_electrique               1000 non-null   int64  
 7   boite_preferee                   1000 non-null   int64  
 8   importance_consommation          1000 non-null   float64
 9   importance_performance           1000 non-null   float64
 10  importance_confort               1000 non-null   float64
 11  importance_prix                  1000 non-null   float64
 12  min_required_seats   

In [217]:
users.head()

,age,sexe,nombre_enfants,revenu_mensuel,budget_max,distance_journaliere_km,prefere_electrique,boite_preferee,importance_consommation,importance_performance,...,energie_electrique,energie_essence,energie_hybride,usage_autoroute,usage_campagne,usage_désert,usage_montagne,usage_plage,usage_sport,usage_ville
0,0.930233,0,0.00,0.029596,0.000000,0.889655,0,1,0.00,0.0,...,0,0,1,0,0,0,0,1,0,0
1,0.813953,0,0.50,0.130866,0.071068,0.593103,0,0,0.50,0.5,...,0,0,0,0,0,0,0,0,1,0
2,0.116279,1,0.00,0.012338,0.000000,0.337931,0,1,1.00,0.5,...,0,0,1,0,0,0,0,1,0,0
3,0.046512,0,1.00,0.303166,0.162986,0.124138,1,1,0.50,0.5,...,1,0,0,0,0,1,0,0,0,0
4,0.883721,0,0.25,0.078519,0.063379,0.468966,1,1,0.25,0.0,...,1,0,0,0,0,0,1,0,0,0


In [218]:
users.select_dtypes(include=["object"]).columns


Index([], dtype='object')

###  # Save

In [181]:
users.to_csv("../data/users_processed.csv", index=False)

# II/ Cars preprocessing

In [3]:
cars = pd.read_csv("../data/cars_versions_specs.csv")

In [22]:
# Filter rows where caractéristiques_carrosserie == "Coupé"
coupe_prices = cars.loc[cars["caractéristiques_carrosserie"] == "Coupé", "price"]

print(coupe_prices)

38     366900
332    340000
436    845000
Name: price, dtype: int64


In [16]:
count_coupe = (cars["caractéristiques_carrosserie"] == "Coupé").sum()
print(count_coupe)

3


In [220]:
cars.columns

Index(['brand', 'model', 'version', 'price', 'caractéristiques_disponibilité',
       'caractéristiques_carrosserie', 'caractéristiques_garantie',
       'caractéristiques_nombre_de_places',
       'caractéristiques_nombre_de_portes', 'motorisation_nombre_de_cylindres',
       'motorisation_energie', 'motorisation_puissance_fiscale',
       'motorisation_puissance_(ch.din)', 'transmission_boîte',
       'transmission_nombre_de_rapports', 'transmission_transmission',
       'consommation_consommation_mixte',
       'aides_à_la_conduite_aide_au_stationnement',
       'equipements_extérieurs_jantes', 'audio_et_communication_ecran_central',
       'equipements_intérieurs_sellerie', 'equipements_intérieurs_volant',
       'equipements_intérieurs_toit',
       'equipements_fonctionnels_frein_de_stationnement',
       'equipements_fonctionnels_modes_de_conduite',
       'performances_autonomie_électrique_(wltp)',
       'consommation_consommation_électrique_(wltp)',
       'consommation_temps

In [221]:
cars.head()

,brand,model,version,price,caractéristiques_disponibilité,caractéristiques_carrosserie,caractéristiques_garantie,caractéristiques_nombre_de_places,caractéristiques_nombre_de_portes,motorisation_nombre_de_cylindres,...,audio_et_communication_ecran_central,equipements_intérieurs_sellerie,equipements_intérieurs_volant,equipements_intérieurs_toit,equipements_fonctionnels_frein_de_stationnement,equipements_fonctionnels_modes_de_conduite,performances_autonomie_électrique_(wltp),consommation_consommation_électrique_(wltp),consommation_temps_de_recharge_normale_(ac),consommation_temps_de_recharge_rapide_(dc)
0,Alfa-Romeo,Alfa Romeo Giulia,Alfa Romeo Giulia,198000,Indisponible,Berline,3 ans,5,4,4.0,...,7 pouces,Cuir,Cuir | Multi-fonctions | Avec palettes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alfa-Romeo,Alfa Romeo Stelvio,Alfa Romeo Stelvio,265000,Indisponible,SUV,3 ans,5,5,4.0,...,Tactile | 6.5 pouces,Cuir,Cuir | Chauffant | Multi-fonctions | Avec pale...,Ouvrant | Panoramique | + 7000 DT,NaN,NaN,NaN,NaN,NaN,NaN
2,Audi,Audi A3 Sportback,35 TFSI Iconic Line,139990,Disponible,Compacte,3 ans,5,5,4.0,...,Tactile | 10.1 pouces,Tissu,Cuir | 3 branches | Multi-fonctions | Avec pal...,NaN,Éléctrique,NaN,NaN,NaN,NaN,NaN
3,Audi,Audi A3 Sportback,35 TFSI Tech Line,154000,Disponible,Compacte,3 ans,5,5,4.0,...,Tactile | 10.1 pouces,Tissu,Cuir | 3 branches | Multi-fonctions | Avec pal...,Panoramique | En verre,Éléctrique,NaN,NaN,NaN,NaN,NaN
4,Audi,Audi A3 Sportback,35 TFSI S-Line,179990,Disponible,Compacte,3 ans,5,5,4.0,...,Tactile | 10.1 pouces,Similicuir/Tissu,Cuir | 3 branches | Multi-fonctions | Sport | ...,Panoramique | En verre,Éléctrique,NaN,NaN,NaN,NaN,NaN


In [222]:
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 29 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   brand                                            549 non-null    object 
 1   model                                            549 non-null    object 
 2   version                                          549 non-null    object 
 3   price                                            549 non-null    int64  
 4   caractéristiques_disponibilité                   549 non-null    object 
 5   caractéristiques_carrosserie                     549 non-null    object 
 6   caractéristiques_garantie                        549 non-null    object 
 7   caractéristiques_nombre_de_places                549 non-null    int64  
 8   caractéristiques_nombre_de_portes                549 non-null    object 
 9   motorisation_nombre_de_cylindres

###  # Basic Cleaning

In [223]:
cars.dtypes

brand                                               object
model                                               object
version                                             object
price                                                int64
caractéristiques_disponibilité                      object
caractéristiques_carrosserie                        object
caractéristiques_garantie                           object
caractéristiques_nombre_de_places                    int64
caractéristiques_nombre_de_portes                   object
motorisation_nombre_de_cylindres                   float64
motorisation_energie                                object
motorisation_puissance_fiscale                     float64
motorisation_puissance_(ch.din)                      int64
transmission_boîte                                  object
transmission_nombre_de_rapports                    float64
transmission_transmission                           object
consommation_consommation_mixte                    float

In [224]:
# Standardize text
text_cols = cars.select_dtypes(include="object").columns

for col in text_cols:
    cars[col] = cars[col].str.strip().str.lower()

In [225]:
def extract_doors(value):
    match = re.search(r'\d+', str(value))
    if match:
        return int(match.group())
    return None

cars["caractéristiques_nombre_de_portes"] = cars["caractéristiques_nombre_de_portes"].apply(extract_doors)


In [226]:
equip_cols = [
    "aides_à_la_conduite_aide_au_stationnement",
    "equipements_extérieurs_jantes",
    "audio_et_communication_ecran_central",
    "equipements_intérieurs_sellerie",
    "equipements_intérieurs_volant",
    "equipements_intérieurs_toit",
    "equipements_fonctionnels_frein_de_stationnement",
    "equipements_fonctionnels_modes_de_conduite",
]

for col in equip_cols:
    cars[col] = cars[col].notna().astype(int)

In [227]:
cars.head()

,brand,model,version,price,caractéristiques_disponibilité,caractéristiques_carrosserie,caractéristiques_garantie,caractéristiques_nombre_de_places,caractéristiques_nombre_de_portes,motorisation_nombre_de_cylindres,...,audio_et_communication_ecran_central,equipements_intérieurs_sellerie,equipements_intérieurs_volant,equipements_intérieurs_toit,equipements_fonctionnels_frein_de_stationnement,equipements_fonctionnels_modes_de_conduite,performances_autonomie_électrique_(wltp),consommation_consommation_électrique_(wltp),consommation_temps_de_recharge_normale_(ac),consommation_temps_de_recharge_rapide_(dc)
0,alfa-romeo,alfa romeo giulia,alfa romeo giulia,198000,indisponible,berline,3 ans,5,4,4.0,...,1,1,1,0,0,0,NaN,NaN,NaN,NaN
1,alfa-romeo,alfa romeo stelvio,alfa romeo stelvio,265000,indisponible,suv,3 ans,5,5,4.0,...,1,1,1,1,0,0,NaN,NaN,NaN,NaN
2,audi,audi a3 sportback,35 tfsi iconic line,139990,disponible,compacte,3 ans,5,5,4.0,...,1,1,1,0,1,0,NaN,NaN,NaN,NaN
3,audi,audi a3 sportback,35 tfsi tech line,154000,disponible,compacte,3 ans,5,5,4.0,...,1,1,1,1,1,0,NaN,NaN,NaN,NaN
4,audi,audi a3 sportback,35 tfsi s-line,179990,disponible,compacte,3 ans,5,5,4.0,...,1,1,1,1,1,0,NaN,NaN,NaN,NaN


###  # Handling Missing Values

In [228]:
# motorisation_nombre_de_cylindres

mode_value = cars["motorisation_nombre_de_cylindres"].mode()[0]
cars["motorisation_nombre_de_cylindres"].fillna(mode_value, inplace=True)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_20308\4162013396.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cars["motorisation_nombre_de_cylindres"].fillna(mode_value, inplace=True)


In [229]:
# motorisation_puissance_fiscale

cars["motorisation_puissance_fiscale"].fillna(
    cars["motorisation_puissance_fiscale"].median(),
    inplace=True
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20308\2367400932.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cars["motorisation_puissance_fiscale"].fillna(


In [230]:
# transmission_nombre_de_rapports

cars["transmission_nombre_de_rapports"].fillna(
    cars["transmission_nombre_de_rapports"].median(),
    inplace=True
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20308\313864469.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cars["transmission_nombre_de_rapports"].fillna(


In [231]:
# consommation_consommation_mixte

# electric → fill with 0
# others → median

cars["is_electric"] = cars["motorisation_energie"].str.contains("elect", case=False, na=False).astype(int)

cars.loc[cars["is_electric"] == 1, "consommation_consommation_mixte"] = 0

cars["consommation_consommation_mixte"].fillna(
    cars["consommation_consommation_mixte"].median(),
    inplace=True
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20308\2220595533.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cars["consommation_consommation_mixte"].fillna(


In [232]:
# Electric-specific columns

ev_cols = [
    "performances_autonomie_électrique_(wltp)",
    "consommation_consommation_électrique_(wltp)",
    "consommation_temps_de_recharge_normale_(ac)",
    "consommation_temps_de_recharge_rapide_(dc)"
]

cars[ev_cols] = cars[ev_cols].fillna(0)

###  # Feature Engineering

In [233]:
# energie 

# cars["is_electric"] = cars["motorisation_energie"].str.contains("elect", case=False, na=False).astype(int)

cars["is_hybrid"] = cars["motorisation_energie"].str.contains("hybrid", case=False, na=False).astype(int)

cars["is_diesel"] = cars["motorisation_energie"].str.contains("diesel", case=False, na=False).astype(int)

cars["is_essence"] = cars["motorisation_energie"].str.contains("essence", case=False, na=False).astype(int)

In [234]:
cars["is_powerful"] = (
    cars["motorisation_puissance_(ch.din)"] > 150
).astype(int)

In [235]:
cars["is_economic"] = (
    cars["consommation_consommation_mixte"] < 6
).astype(int)

In [236]:
cars["is_family"] = (
    cars["caractéristiques_nombre_de_places"] >= 5
).astype(int)

###  # Encoding

In [237]:
# Brand category

premium = [
"Audi","Bmw","Mercedes-Benz","Jaguar","Porsche",
"Land-Rover","Mini","Volvo"
]

midrange = [
"Toyota","Volkswagen","Peugeot","Renault","Hyundai",
"Kia","Honda","Skoda","Seat","Jeep","Ford",
"Citroen","Nissan","Mg","Cupra","Suzuki"
]

def brand_category(brand):

    if brand in premium:
        return "premium"

    elif brand in midrange:
        return "midrange"

    else:
        return "budget"

cars["brand_category"] = cars["brand"].apply(brand_category)

cars = pd.get_dummies(cars, columns=["brand_category"])

In [238]:
# Carrosserie

cars = pd.get_dummies(
    cars,
    columns=["caractéristiques_carrosserie"],
    prefix="body"
)

In [239]:
# transmission_boîte

cars = pd.get_dummies( cars, columns=['transmission_boîte'], drop_first=False )

In [240]:
# transmission_transmission

cars = pd.get_dummies( cars, columns=['transmission_transmission'], drop_first=False )

In [241]:
# Convert all bool columns to int

bool_cols = cars.select_dtypes(include="bool").columns
cars[bool_cols] = cars[bool_cols].astype(int)

###  # Scaling

In [242]:
# Numerical columns to scale (continuous values)
numerical_cols = [
    "price",
    "motorisation_puissance_(ch.din)",
    "consommation_consommation_mixte",
    "motorisation_puissance_fiscale",
    "caractéristiques_nombre_de_places",
    "transmission_nombre_de_rapports",
    "caractéristiques_nombre_de_portes",  
    "motorisation_nombre_de_cylindres",
    "performances_autonomie_électrique_(wltp)",
    "consommation_consommation_électrique_(wltp)", 
    "consommation_temps_de_recharge_normale_(ac)",
    "consommation_temps_de_recharge_rapide_(dc)"
]

# Binary/categorical columns (should NOT be scaled)
binary_patterns = [
    'aides_', 'equipements_', 'is_', 'brand_', 'body_', 'transmission_'
]

# Get all binary columns
binary_cols = []
for col in cars.columns:
    if any(pattern in col for pattern in binary_patterns):
        if col not in numerical_cols:  # Make sure it's not in numerical
            binary_cols.append(col)

print(f"Found {len(numerical_cols)} numerical columns")
print(f"Found {len(binary_cols)} binary/categorical columns")

# Scale only numerical columns
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# Store original values for binary columns
cars[numerical_cols] = scaler.fit_transform(cars[numerical_cols])

# Verify binary columns are still 0/1
for col in binary_cols[:5]:
    unique_vals = cars[col].unique()
    print(f"{col}: values {unique_vals}")

print("\nScaling complete - binary features preserved as 0/1")

Found 12 numerical columns
Found 30 binary/categorical columns
aides_à_la_conduite_aide_au_stationnement: values [1 0]
equipements_extérieurs_jantes: values [1 0]
equipements_intérieurs_sellerie: values [1]
equipements_intérieurs_volant: values [1 0]
equipements_intérieurs_toit: values [0 1]

Scaling complete - binary features preserved as 0/1


###  # Drop

In [243]:
drop_cols = [
    "brand",
    "model",
    "version",
    "caractéristiques_disponibilité",
    "caractéristiques_garantie",
    "motorisation_energie",	
]

cars = cars.drop(drop_cols, axis=1)

###  # Final Check

In [244]:
cars.isna().sum().sum()

0

In [246]:
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 43 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   price                                            549 non-null    float64
 1   caractéristiques_nombre_de_places                549 non-null    float64
 2   caractéristiques_nombre_de_portes                549 non-null    float64
 3   motorisation_nombre_de_cylindres                 549 non-null    float64
 4   motorisation_puissance_fiscale                   549 non-null    float64
 5   motorisation_puissance_(ch.din)                  549 non-null    float64
 6   transmission_nombre_de_rapports                  549 non-null    float64
 7   consommation_consommation_mixte                  549 non-null    float64
 8   aides_à_la_conduite_aide_au_stationnement        549 non-null    int32  
 9   equipements_extérieurs_jantes   

In [247]:
cars.head()

,price,caractéristiques_nombre_de_places,caractéristiques_nombre_de_portes,motorisation_nombre_de_cylindres,motorisation_puissance_fiscale,motorisation_puissance_(ch.din),transmission_nombre_de_rapports,consommation_consommation_mixte,aides_à_la_conduite_aide_au_stationnement,equipements_extérieurs_jantes,...,body_minibus,body_monospace,body_pick up,body_suv,body_utilitaire,transmission_boîte_automatique,transmission_boîte_manuelle,transmission_transmission_intégrale,transmission_transmission_propulsion,transmission_transmission_traction
0,0.218618,0.181818,0.666667,0.333333,0.322581,0.230679,0.777778,0.472,1,1,...,0,0,0,0,0,1,0,0,1,0
1,0.299534,0.181818,1.000000,0.333333,0.516129,0.324356,0.777778,0.560,1,1,...,0,0,0,1,0,1,0,1,0,0
2,0.148559,0.181818,1.000000,0.333333,0.225806,0.172131,0.777778,0.432,1,1,...,0,0,0,0,0,1,0,0,0,1
3,0.165479,0.181818,1.000000,0.333333,0.225806,0.172131,0.777778,0.432,1,1,...,0,0,0,0,0,1,0,0,0,1
4,0.196867,0.181818,1.000000,0.333333,0.225806,0.172131,0.777778,0.432,1,1,...,0,0,0,0,0,1,0,0,0,1


###  # Save

In [248]:
cars.to_csv("../data/cars_processed.csv", index=False)

# III/ Preprocessed datasets

In [182]:
processed_users = pd.read_csv("../data/users_processed.csv")

In [183]:
processed_users.head()

,age,sexe,nombre_enfants,revenu_mensuel,budget_max,distance_journaliere_km,prefere_electrique,boite_preferee,importance_consommation,importance_performance,...,energie_electrique,energie_essence,energie_hybride,usage_autoroute,usage_campagne,usage_désert,usage_montagne,usage_plage,usage_sport,usage_ville
0,0.930233,0,0.00,0.029596,0.000000,0.889655,0,1,0.00,0.0,...,0,0,1,0,0,0,0,1,0,0
1,0.813953,0,0.50,0.130866,0.071068,0.593103,0,0,0.50,0.5,...,0,0,0,0,0,0,0,0,1,0
2,0.116279,1,0.00,0.012338,0.000000,0.337931,0,1,1.00,0.5,...,0,0,1,0,0,0,0,1,0,0
3,0.046512,0,1.00,0.303166,0.162986,0.124138,1,1,0.50,0.5,...,1,0,0,0,0,1,0,0,0,0
4,0.883721,0,0.25,0.078519,0.063379,0.468966,1,1,0.25,0.0,...,1,0,0,0,0,0,1,0,0,0


In [185]:
processed_users.columns

Index(['age', 'sexe', 'nombre_enfants', 'revenu_mensuel', 'budget_max',
       'distance_journaliere_km', 'prefere_electrique', 'boite_preferee',
       'importance_consommation', 'importance_performance',
       'importance_confort', 'importance_prix', 'min_required_seats',
       'has_children', 'long_commute', 'needs_awd',
       'situation_familiale_célibataire', 'situation_familiale_divorcé(e)',
       'situation_familiale_marié(e)', 'carrosserie_berline',
       'carrosserie_cabriolet', 'carrosserie_citadine', 'carrosserie_compacte',
       'carrosserie_coupé', 'carrosserie_minibus', 'carrosserie_monospace',
       'carrosserie_pick up', 'carrosserie_suv', 'carrosserie_utilitaire',
       'energie_diesel', 'energie_electrique', 'energie_essence',
       'energie_hybride', 'usage_autoroute', 'usage_campagne', 'usage_désert',
       'usage_montagne', 'usage_plage', 'usage_sport', 'usage_ville'],
      dtype='object')

In [166]:
processed_cars = pd.read_csv("../data/cars_processed.csv")

In [167]:
processed_cars.head()

,price,caractéristiques_nombre_de_places,caractéristiques_nombre_de_portes,motorisation_nombre_de_cylindres,motorisation_puissance_fiscale,motorisation_puissance_(ch.din),transmission_nombre_de_rapports,consommation_consommation_mixte,aides_à_la_conduite_aide_au_stationnement,equipements_extérieurs_jantes,...,body_minibus,body_monospace,body_pick up,body_suv,body_utilitaire,transmission_boîte_automatique,transmission_boîte_manuelle,transmission_transmission_intégrale,transmission_transmission_propulsion,transmission_transmission_traction
0,0.238242,-0.007999,-0.708846,0.205576,0.608088,0.054698,0.887389,0.319592,1,1,...,0,0,0,0,0,1,0,0,1,0
1,0.799234,-0.007999,0.562907,0.205576,2.002963,0.720724,0.887389,0.706137,1,1,...,0,0,0,1,0,1,0,1,0,0
2,-0.247477,-0.007999,0.562907,0.205576,-0.089350,-0.361567,0.887389,0.143890,1,1,...,0,0,0,0,0,1,0,0,0,1
3,-0.130171,-0.007999,0.562907,0.205576,-0.089350,-0.361567,0.887389,0.143890,1,1,...,0,0,0,0,0,1,0,0,0,1
4,0.087444,-0.007999,0.562907,0.205576,-0.089350,-0.361567,0.887389,0.143890,1,1,...,0,0,0,0,0,1,0,0,0,1


In [168]:
processed_cars['performances_autonomie_électrique_(wltp)'].unique()

array([-4.78567920e-01,  3.23138024e+00,  2.93370238e+00,  2.99434046e+00,
        2.20604538e+00,  5.24716724e-01,  1.17519798e+00, -9.26892106e-02,
        1.82938439e-01,  7.26873790e-02,  6.23942677e-01,  8.99570327e-01,
        2.85869551e-02,  6.16622730e-02, -1.00009158e-02,  6.53674316e-03,
        1.27812909e-01,  2.82896387e+00,  1.93593029e+00,  1.99656837e+00,
        2.60846175e+00,  1.75952859e+00,  3.01087812e+00,  7.89319267e-01,
       -1.75377505e-01,  1.39570010e+00,  1.83670433e+00,  1.75618491e-02,
        2.38795963e+00,  9.27133092e-01,  1.00982139e+00,  4.03440558e-01,
        4.58566088e-01,  1.02419017e-03,  8.44444797e-01,  8.27907138e-01,
        1.41775031e+00,  1.31301180e+00,  1.86426710e+00,  2.05720645e+00,
        1.26891138e+00,  1.63825243e+00,  2.14540730e+00,  2.76832579e+00,
        2.90613961e+00,  2.73525047e+00,  2.22258304e+00,  3.13215429e+00,
        2.69115005e+00,  2.00208092e+00,  2.58089899e+00,  2.30527134e+00,
        1.92490518e+00,  

In [131]:
processed_cars.columns

Index(['price', 'caractéristiques_nombre_de_places',
       'caractéristiques_nombre_de_portes', 'motorisation_nombre_de_cylindres',
       'motorisation_puissance_fiscale', 'motorisation_puissance_(ch.din)',
       'transmission_nombre_de_rapports', 'consommation_consommation_mixte',
       'aides_à_la_conduite_aide_au_stationnement',
       'equipements_extérieurs_jantes', 'audio_et_communication_ecran_central',
       'equipements_intérieurs_sellerie', 'equipements_intérieurs_volant',
       'equipements_intérieurs_toit',
       'equipements_fonctionnels_frein_de_stationnement',
       'equipements_fonctionnels_modes_de_conduite',
       'performances_autonomie_électrique_(wltp)',
       'consommation_consommation_électrique_(wltp)',
       'consommation_temps_de_recharge_normale_(ac)',
       'consommation_temps_de_recharge_rapide_(dc)', 'is_electric',
       'is_hybrid', 'is_diesel', 'is_essence', 'is_powerful', 'is_economic',
       'is_family', 'brand_category_budget', 'body_ber